# Task 4 : Naïve Bayesian Classifier

    ## Requirements
    - Dataset: quiz scores (Q1–Q9) + course result (`Rank`: P = Pass, F = Fail).
    - Scores are **continuous** → discretise first, then classify.
    - Manually implement the **Naïve Bayes** algorithm (no sklearn).
    - Compute **accuracy** on the full dataset.
    - OOP model, compact and readable code.

## Import Libraries

In [13]:
!pip install prettytable -q

import pandas as pd
import numpy as np
from abc import ABC, abstractmethod
from collections import defaultdict
from prettytable import PrettyTable

## Architecture Overview
```
State            – one labelled data instance (feature dict + true label)
Problem          – loads CSV; preprocesses; discretises; builds States
SearchStrategy   – abstract base (ABC)
  └─ NaiveBayes  – manual implementation with Laplace smoothing
Solution         – binds Problem + Strategy; solve / evaluate / report
```

## State Class

In [14]:
class State:
    """
    One labelled data instance.

    Attributes
    ----------
    features : dict  – {feature_name: discretised_bin_label}
    label    : str   – true class ("P" or "F"), None for unlabelled
    """

    def __init__(self, features: dict, label: str = None):
        self.features = features
        self.label    = label

    def __repr__(self) -> str:
        return f"State(label={self.label!r}, features={self.features})"

## Problem Class

In [15]:
class Problem:
    """
    Dataset lifecycle: from_file → preprocess → discretize → get_states.

    Attributes (after discretize())
    --------------------------------
    features  : list[str]   – feature column names (Q1 … Q9)
    classes   : ndarray     – unique class labels
    states    : list[State]
    bins_info : dict        – {feature: [(label, lo, hi), …]}
    """

    def __init__(self, filepath: str):
        self.filepath  = filepath
        self.raw_df    = None
        self.disc_df   = None
        self.features  = []
        self.target    = "Rank"
        self.classes   = None
        self.states    = []
        self.bins_info = {}

    # ── factory ──────────────────────────────────────────────────────────────
    @classmethod
    def from_file(cls, filepath: str) -> "Problem":
        """Construct, load, and preprocess in one call."""
        prob = cls(filepath)
        prob._load()
        prob._preprocess()
        return prob

    # ── pipeline (private) ────────────────────────────────────────────────────
    def _load(self):
        self.raw_df = pd.read_csv(self.filepath)

    def _preprocess(self):
        df = self.raw_df.copy()
        df = df.drop(columns=["#"], errors="ignore")
        df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
        self.features = [c for c in df.columns if c != self.target]
        self.classes  = df[self.target].unique()
        self.raw_df   = df

    # ── public ────────────────────────────────────────────────────────────────
    def discretize(self, bins: int = 3) -> "Problem":
        """Discretise each feature into *bins* equal-width bins."""
        df = self.raw_df.copy()
        self.bins_info = {}

        for feat in self.features:
            lo, hi = df[feat].min(), df[feat].max()
            if lo == hi:
                df[feat]             = "bin_0"
                self.bins_info[feat] = [("bin_0", lo, hi)]
                continue

            edges       = np.linspace(lo, hi, bins + 1)
            edges[-1]  += 0.001
            labels      = [f"bin_{i}" for i in range(bins)]
            df[feat], returned_edges = pd.cut(
                self.raw_df[feat], bins=edges, labels=labels,
                include_lowest=True, retbins=True,
            )
            self.bins_info[feat] = [
                (f"bin_{i}", returned_edges[i], returned_edges[i+1])
                for i in range(bins)
            ]

        self.disc_df = df
        self.states  = [
            State({f: row[f] for f in self.features}, label=str(row[self.target]))
            for _, row in df.iterrows()
        ]
        return self

    def get_states(self) -> list:
        if not self.states:
            raise RuntimeError("Call discretize() before get_states().")
        return self.states

    def print_bins(self) -> None:
        """Display bin ranges for each feature using PrettyTable."""
        n_bins = len(next(iter(self.bins_info.values())))

        table = PrettyTable()
        table.title       = f"Discretisation — {n_bins} Equal-Width Bins per Feature"
        table.field_names = ["Feature"] + [f"bin_{i}" for i in range(n_bins)]
        table.align       = "c"

        for feat, ranges in self.bins_info.items():
            row = [feat] + [f"[{lo:.2f}, {hi:.2f})" for _, lo, hi in ranges]
            table.add_row(row)

        print(table)

## SearchStrategy (ABC) & NaiveBayes

In [16]:
class SearchStrategy(ABC):
    """Abstract classifier – subclasses implement train() and predict()."""

    @abstractmethod
    def train(self, states: list, classes, features: list) -> "SearchStrategy":
        """Learn parameters from states. Returns self (fluent)."""

    @abstractmethod
    def predict(self, state: State) -> str:
        """Return predicted class label for one state."""


class NaiveBayes(SearchStrategy):
    """
    Manual Naïve Bayes with Laplace smoothing.

    Model
    -----
    P(C | x₁…xₙ) ∝ P(C) · ∏ P(xᵢ | C)

    Computation is done in log-space to avoid float underflow.
    Laplace smoothing handles unseen (feature, value) pairs.
    """

    def __init__(self):
        self.prior      = {}          # {class: log P(C)}
        self.likelihood = defaultdict(lambda: defaultdict(dict))
        # likelihood[class][feature][bin_label] = log P(val | class)
        self._features  = []
        self._classes   = []

    def train(self, states: list, classes, features: list) -> "NaiveBayes":
        self._features = list(features)
        self._classes  = list(classes)
        n              = len(states)

        # ── priors ───────────────────────────────────────────────────────────
        class_cnt = defaultdict(int)
        for s in states: class_cnt[s.label] += 1
        self.prior = {c: np.log(class_cnt[c] / n) for c in self._classes}

        # ── likelihoods ───────────────────────────────────────────────────────
        for c in self._classes:
            class_states = [s for s in states if s.label == c]
            n_c          = len(class_states)
            for feat in self._features:
                val_cnt = defaultdict(int)
                for s in class_states: val_cnt[s.features[feat]] += 1
                for val, cnt in val_cnt.items():
                    self.likelihood[c][feat][val] = np.log(cnt / n_c)
        return self

    def predict(self, state: State) -> str:
        scores = {}
        for c in self._classes:
            log_prob = self.prior[c]
            for feat in self._features:
                val = state.features[feat]
                if val in self.likelihood[c][feat]:
                    log_prob += self.likelihood[c][feat][val]
                else:
                    # Laplace smoothing for unseen value
                    n_known   = max(1, len(self.likelihood[c][feat]))
                    log_prob += np.log(1.0 / (n_known + 1))
            scores[c] = log_prob
        return max(scores, key=scores.get)

## Solution Class

In [17]:
class Solution:
    """
    Single entry point – binds Problem + Strategy.

    Usage
    -----
    problem  = Problem.from_file("task4_data.csv")
    problem.discretize(bins=3)
    strategy = NaiveBayes()
    sol      = Solution(problem, strategy)
    sol.solve()          # trains the model
    sol.print_report()   # accuracy + confusion matrix
    """

    def __init__(self, problem: Problem, strategy: SearchStrategy):
        self.problem  = problem
        self.strategy = strategy
        self.accuracy = None

    def solve(self) -> "Solution":
        """Train on all states, then compute in-sample accuracy."""
        states = self.problem.get_states()
        self.strategy.train(states, self.problem.classes, self.problem.features)
        correct       = sum(1 for s in states if self.strategy.predict(s) == s.label)
        self.accuracy = correct / len(states)
        return self

    def predict_one(self, state) -> str:
        return self.strategy.predict(state)

    def print_report(self) -> None:
        """Print accuracy summary and confusion matrix using PrettyTable."""
        states = self.problem.get_states()
        tp = fp = tn = fn = 0
        for s in states:
            pred, true_ = self.strategy.predict(s), s.label
            if   pred == "P" and true_ == "P": tp += 1
            elif pred == "P" and true_ == "F": fp += 1
            elif pred == "F" and true_ == "F": tn += 1
            else:                               fn += 1

        # ── summary table ────────────────────────────────────────────────────
        summary = PrettyTable()
        summary.title        = "Naïve Bayes — Classification Report"
        summary.field_names  = ["Metric", "Value"]
        summary.align        = "l"
        summary.add_rows([
            ["Total instances",    len(states)],
            ["Correct predictions",tp + tn],
            ["Wrong predictions",  fp + fn],
            ["Accuracy",           f"{self.accuracy:.4f}  ({self.accuracy*100:.2f}%)"],
        ])
        print(summary)

        # ── confusion matrix ──────────────────────────────────────────────────
        cm = PrettyTable()
        cm.title            = "Confusion Matrix"
        cm.field_names      = ["", "Predicted P", "Predicted F"]
        cm.align            = "c"
        cm.add_row(["Actual P", tp, fn])
        cm.add_row(["Actual F", fp, tn])
        print(cm)

## Run

In [18]:
def main():
    # 1. Load & preprocess
    problem = Problem.from_file("task4_data.csv")

    # 2. Discretise
    bins = 3
    print(f"Discretising with {bins} equal-width bins …")
    problem.discretize(bins=bins)
    problem.print_bins()

    # 3. Train & evaluate
    strategy = NaiveBayes()
    sol      = Solution(problem, strategy)
    print("Training Naïve Bayes …")
    sol.solve()
    sol.print_report()

main()

Discretising with 3 equal-width bins …
+-------------------------------------------------------+
|    Discretisation — 3 Equal-Width Bins per Feature    |
+---------+--------------+--------------+---------------+
| Feature |    bin_0     |    bin_1     |     bin_2     |
+---------+--------------+--------------+---------------+
|    Q1   | [0.00, 2.33) | [2.33, 4.67) |  [4.67, 7.00) |
|    Q2   | [0.00, 3.33) | [3.33, 6.67) | [6.67, 10.00) |
|    Q3   | [0.00, 3.33) | [3.33, 6.67) | [6.67, 10.00) |
|    Q4   | [0.00, 3.33) | [3.33, 6.67) | [6.67, 10.00) |
|    Q5   | [0.00, 3.00) | [3.00, 6.00) |  [6.00, 9.00) |
|    Q6   | [0.00, 3.33) | [3.33, 6.67) | [6.67, 10.00) |
|    Q7   | [0.00, 2.67) | [2.67, 5.33) |  [5.33, 8.00) |
|    Q8   | [0.00, 3.10) | [3.10, 6.20) |  [6.20, 9.30) |
|    Q9   | [0.00, 3.33) | [3.33, 6.67) | [6.67, 10.00) |
+---------+--------------+--------------+---------------+
Training Naïve Bayes …
+----------------------------------------+
|  Naïve Bayes — Classifi